# Superficie extendida — particiones 25,27,29 de 40

Replay de verificación de `garrido_transfer_confirmation_v2_ext`: cada rebanada se recalcula celda a
celda y se compara **bit a bit** contra el caché sellado. No abre semillas, no entrena, no adjudica
nada científico — es procedencia.

Las cuarenta particiones están repartidas: local 0-23, este par de kernels 24-29, VPS 30-39. Son
disjuntas a propósito; dos pools sobre particiones solapadas duplican trabajo en vez de repartirlo.

Las rebanadas ya terminadas viajan en el repositorio, así que este kernel se las salta y sólo calcula
lo que falta.

Regla de tolerancia: `docs/ENMIENDA_TOLERANCIA_EQUIVALENCIA_CROSS_PLATFORM_2026-08-08.md`,
congelada antes de ver ningún resultado de la extendida.


In [ ]:
# 1) Repo y dependencias
import os, subprocess, sys, time, json, shutil, platform
from pathlib import Path

ROOT = Path('/kaggle/working/scres-ia')
if not ROOT.exists():
    subprocess.check_call(['git', 'clone', '--depth', '1', '--branch', 'codex/expanded-contract-comparators-v2',
                           'https://github.com/Thom-320/scres-ia.git', str(ROOT)])
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'simpy>=4.1', 'numpy', 'pandas', 'scipy', 'scikit-learn'])

# The host identity travels with the result: a bit-exact claim across architectures is worth
# nothing if nobody recorded which architectures.
env = {'platform': platform.platform(), 'machine': platform.machine(),
       'python': platform.python_version(), 'cpus': os.cpu_count(),
       'commit': subprocess.check_output(['git', 'rev-parse', 'HEAD']).decode().strip()}
print(json.dumps(env, indent=1))
Path('/kaggle/working/host_env_scresia-ext-surface-b.json').write_text(json.dumps(env, indent=1))
done = len(list(Path('results/frozen_path_equivalence_v2/shards').glob('ext__*.json')))
print(f'rebanadas ext ya en el repo: {done}/360')

In [ ]:
# 2) Una partición por proceso. Independientes: en Linux esto es fork y no hay
#    ProcessPoolExecutor de por medio, que es lo que colgaba en macOS.
parts = [25, 27, 29]
procs = []
for p in parts:
    log = open(f'/kaggle/working/ext_{p}.log', 'w')
    procs.append(subprocess.Popen(
        [sys.executable, '-u', 'scripts/verify_frozen_path_equivalence_v2.py',
         '--phase', 'surface', '--surface', 'ext', '--of', '40', '--shards', str(p)],
        stdout=log, stderr=subprocess.STDOUT))
print(f'lanzados {len(procs)} procesos sobre particiones {parts}', flush=True)

t0 = time.time()
while any(pr.poll() is None for pr in procs):
    time.sleep(120)
    n = len(list(Path('results/frozen_path_equivalence_v2/shards').glob('ext__*.json')))
    print(f'  [{(time.time()-t0)/60:.0f} min] ext en disco: {n}/360', flush=True)
print('codigos de salida:', [pr.returncode for pr in procs])

In [ ]:
# 3) Sólo las rebanadas nuevas vuelven, con su diagnóstico de diferencias.
out = Path('/kaggle/working/ext_shards_scresia-ext-surface-b')
out.mkdir(exist_ok=True)
mismatched = []
for f in Path('results/frozen_path_equivalence_v2/shards').glob('ext__*.json'):
    d = json.loads(f.read_text())
    if d.get('mismatches'):
        mismatched.append({'file': f.name, 'mismatches': d['mismatches'],
                           'max_abs_delta': d.get('max_abs_delta')})
    shutil.copy(f, out / f.name)
print(f'rebanadas empaquetadas: {len(list(out.glob("*.json")))}')
print(f'con diferencias: {len(mismatched)}')
for m in mismatched[:10]:
    print(' ', m)
shutil.make_archive('/kaggle/working/ext_shards_scresia-ext-surface-b', 'zip', out)
print('ZIP -> /kaggle/working/ext_shards_scresia-ext-surface-b.zip')